# LightRAG Document Ingestion

This notebook implements a lightweight RAG system for processing PDF and text documents using local Ollama embeddings.

## Setup

First, install required packages:

In [ ]:
!pip install numpy requests PyPDF2 tqdm

## Import Dependencies

In [ ]:
import os
from pathlib import Path
from typing import List, Dict, Union
import PyPDF2
import numpy as np
import json
import pickle
from tqdm import tqdm
import requests

## Implement Ollama Embedding Client

In [ ]:
class OllamaEmbedding:
    def __init__(self, model_name: str = "nomic-embed-text", base_url: str = "http://localhost:11434"):
        """Initialize Ollama embedding client."""
        self.model_name = model_name
        self.base_url = base_url
    
    def encode(self, texts: Union[str, List[str]]) -> np.ndarray:
        """Generate embeddings for text using Ollama API."""
        if isinstance(texts, str):
            texts = [texts]
        
        embeddings = []
        for text in texts:
            response = requests.post(
                f"{self.base_url}/api/embeddings",
                json={
                    "model": self.model_name,
                    "prompt": text
                }
            )
            embeddings.append(response.json()['embedding'])
        
        return np.array(embeddings)

## Document Processing Functions

In [ ]:
def read_pdf(file_path: str) -> str:
    """Read content from a PDF file."""
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        text = ''
        for page in pdf_reader.pages:
            text += page.extract_text() + '\n'
    return text

def read_text(file_path: str) -> str:
    """Read content from a text file."""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def chunk_text(text: str, chunk_size: int = 512, overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

## LightRAG Implementation

In [ ]:
class LightRAG:
    def __init__(self, embedding_provider: str = "ollama", 
                 model_name: str = "nomic-embed-text"):
        """Initialize LightRAG with chosen embedding provider."""
        if embedding_provider == "ollama":
            self.model = OllamaEmbedding(model_name)
        else:
            raise ValueError("Unsupported embedding provider")
            
        self.documents = []
        self.embeddings = None
        self.metadata = []
        
    def add_document(self, content: str, metadata: Dict):
        """Add a document and its metadata to the RAG system."""
        chunks = chunk_text(content)
        
        # Create embeddings for chunks
        chunk_embeddings = self.model.encode(chunks)
        
        # Store chunks and their metadata
        if self.embeddings is None:
            self.embeddings = chunk_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, chunk_embeddings])
        
        # Store document chunks and metadata
        for chunk in chunks:
            self.documents.append(chunk)
            self.metadata.append(metadata)
    
    def save(self, directory: str):
        """Save the RAG system to disk."""
        os.makedirs(directory, exist_ok=True)
        
        # Save embeddings
        np.save(os.path.join(directory, 'embeddings.npy'), self.embeddings)
        
        # Save documents and metadata
        with open(os.path.join(directory, 'documents.pkl'), 'wb') as f:
            pickle.dump({'documents': self.documents, 'metadata': self.metadata}, f)

## Process Documents

Now let's process all documents in the specified directory:

In [ ]:
# Initialize LightRAG
rag = LightRAG(embedding_provider="ollama", model_name="nomic-embed-text")

# Directory containing documents
docs_dir = "documents/"

# Process all documents in the directory
for file_path in tqdm(Path(docs_dir).rglob('*')):
    if file_path.suffix.lower() == '.pdf':
        content = read_pdf(str(file_path))
    elif file_path.suffix.lower() == '.txt':
        content = read_text(str(file_path))
    else:
        continue
        
    # Create metadata for the document
    metadata = {
        'filename': file_path.name,
        'path': str(file_path),
        'type': file_path.suffix.lower()
    }
    
    # Add document to RAG
    rag.add_document(content, metadata)

# Save the RAG system
rag.save('lightrag_index')

print("Document processing and indexing complete!")